<a href="https://colab.research.google.com/github/srijabiswas-01/audio_analysis/blob/main/audio_spliter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!apt-get -qq update
!apt-get -qq install -y ffmpeg
!pip -q install demucs soundfile librosa ffmpeg-python

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 32.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.1/87.1 kB 8.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 249.3/249.3 kB 24.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.0/40.0 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.0/76.0 kB 9.0 MB/s eta 0:00:00


In [2]:
!pip install demucs

In [3]:
from google.colab import files
import subprocess
from pathlib import Path
import librosa
import shutil

In [5]:
# Maximum allowed duration (8 minutes = 480 seconds)
MAX_DURATION_SECONDS = 8 * 60

# Upload file
uploaded = files.upload()

input_filename = next(iter(uploaded.keys()))
print("Uploaded:", input_filename)

# Check duration before processing
try:
    duration = librosa.get_duration(path=input_filename)
except TypeError:
    # Fallback for older librosa versions
    y, sr = librosa.load(input_filename, sr=None, mono=True)
    duration = librosa.get_duration(y=y, sr=sr)

minutes = int(duration // 60)
seconds = int(duration % 60)

print(f"Audio Duration: {minutes} min {seconds} sec")

if duration > MAX_DURATION_SECONDS:
    raise ValueError(
        f"❌ Audio is too long ({minutes}m {seconds}s). "
        f"Please upload a file that is 8 minutes or shorter."
    )

print("✅ Duration check passed.")

Saving Ae Ajnabi.mp3 to Ae Ajnabi.mp3
Uploaded: Ae Ajnabi.mp3
Audio Duration: 5 min 40 sec
✅ Duration check passed.


In [8]:
# uploaded = files.upload()
# input_filename = next(iter(uploaded.keys()))
# print("Uploaded:", input_filename)

In [6]:
input_path = Path(input_filename)
converted_path = "input_audio.wav"

subprocess.run([
    "ffmpeg",
    "-y",
    "-i",
    str(input_path),
    "-ar", "44100",
    "-ac", "2",
    converted_path
], check=True)

print("Converted to:", converted_path)

Converted to: input_audio.wav


In [7]:
subprocess.run([
    "python",
    "-m",
    "demucs",
    "--mp3",
    "--filename",
    "{track}/{stem}.{ext}",
    converted_path
], check=True)

print("Separation complete.")

Separation complete.


In [9]:
root = Path("separated/htdemucs/input_audio")
print("Generated files:")
for f in root.glob("*"):
    print("-", f)

Generated files:
- separated/htdemucs/input_audio/bass.mp3
- separated/htdemucs/input_audio/drums.mp3
- separated/htdemucs/input_audio/other.mp3
- separated/htdemucs/input_audio/vocals.mp3


In [10]:
zip_path = shutil.make_archive(
    "demucs_output",
    "zip",
    root_dir=root
)
print(zip_path)

/content/demucs_output.zip


In [11]:
files.download(zip_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>